In [1]:
import re
import compress_fasttext
from tqdm import tqdm
from pathlib import Path
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein
import pymorphy3
from utils import get_concepts

In [2]:
# ! curl -L -o fasttext.bin "https://github.com/avidale/compress-fasttext/releases/download/gensim-4-draft/geowac_tokens_sg_300_5_2020-100K-20K-100.bin"

In [3]:
fasttext = compress_fasttext.models.CompressedFastTextKeyedVectors.load('fasttext.bin')
morph = pymorphy3.MorphAnalyzer()

In [4]:
concepts = get_concepts()
concepts[:3]

[('Q1', 'вселенная'),
 ('Q102836', 'пружина'),
 ('Q104837', 'термодинамическая фаза')]

In [5]:
print('Макисмальное количество слов в понятии (для выбора n-грамм): n =',
      max(map(lambda x: len(x[1].split()), concepts)))

Макисмальное количество слов в понятии (для выбора n-грамм): n = 4


In [6]:
for c in concepts:
    if len(c[1].split()) > 2:
        print(c)

('Q11382', 'закон сохранения энергии')
('Q134465', 'классическая теория тяготения ньютона')
('Q161635', 'вектор угловой скорости')
('Q192704', 'коэффициент полезного действия')
('Q2305665', 'закон сохранения импульса')
('Q317949', 'активность радиоактивного источника')
('Q483948', 'закон сохранения массы')
('Q649036', 'давление электромагнитного излучения')


In [7]:
def read_concept(concept_id: str):
    with open(Path.cwd().parent / 'papers' / f'{concept_id}.md', mode='r', encoding='utf-8') as file:
        return '\n'.join(file.readlines())
    

def write_concept(concept_id: str, article: str):
    with open(Path.cwd().parent / 'papers' / f'{concept_id}.md', mode='w', encoding='utf-8') as file:
        file.write(article)

In [8]:
def norm(text):
    clean = re.sub('[^\w\s]', ' ', text)
    clean = re.sub('  ', ' ', clean)
    words = clean.split()
    parsed_words = [morph.parse(word)[0] for word in words]
    result = (
        ' '.join([word.normal_form for word in parsed_words]),
        [word.tag.POS for word in parsed_words]
    )
    return result

morphed_concepts = []
for concept_id, concept_name in concepts:
    norm_concept, poses = norm(concept_name)
    morphed_concepts.append((
        concept_id,
        norm_concept,
        poses
    ))
morphed_concepts[:4]

[('Q1', 'вселенная', ['NOUN']),
 ('Q102836', 'пружина', ['NOUN']),
 ('Q104837', 'термодинамический фаза', ['ADJF', 'NOUN']),
 ('Q1075', 'цвет', ['NOUN'])]

In [9]:
def find_related_concept(text: str, except_concept: str):
    """
    text - отрывок (одно, два или три слова) из текста,
    которые предположительно являются концептом. Если
    в text есть лишние слова или отсутствуют слова или
    эти слова не являются концептом, то метод вернёт
    Исходную строку. Иначе метод вернёт корректную
    файловую ссылку в формате Markdown на этот отрывок

    Пример:
    >>> find_related_concept("**Сила**")
    [**Сила**](Q11402.md)

    >>> find_related_concept("силы _Лоренца_")
    [силы _Лоренца_](Q172137.md)

    >>> find_related_concept("сила лоренца присутствует")
    сила лоренца присутствует
    """
    original_text = text[:]

    # Заголовок
    if '#' in text:
        return original_text

    # Некорректный отрывок (слова должны идти подряд, а не через запятую, точку)
    words = text.split()
    if any({'.', ',', ':'} & set(word) for word in words[:-1]):
        return original_text
    
    # ---

    # Нормализуем вход и создаем сет слов для жесткой проверки
    clean_text = re.sub(r'[^\w\s]', '', text.lower())
    norm_text, poses_text = norm(clean_text)
    
    # Поиск кандидатов
    results = process.extract(
        norm_text,
        [c[1] for c in morphed_concepts],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60
    )
    
    answer = []
    for matching, _, id in results:
        poses_match = morphed_concepts[id][2][:]
        if len(poses_text) != len(poses_match):
            continue
        norm_text_splited = norm_text.split()

        matching = matching.split()
        new_matching = []
        f_score = 0
        for w_orig, p_orig in zip(norm_text_splited, poses_text):
            flag = False
            for w_match, p_match in zip(matching, poses_match):
                if p_orig != p_match:
                    continue
                dist = Levenshtein.distance(w_orig, w_match) / len(w_orig)
                if dist > 0.3:
                    continue
                f_score += (1 - dist) / len(norm_text_splited)
                flag = True
                idx = matching.index(w_match)
                del matching[idx]
                del poses_match[idx]
                new_matching.append(w_match)
                break
            if not flag:
                break

        if len(new_matching) != len(norm_text_splited):
            continue

        # Оценка сходства
        similarity = fasttext.similarity(norm_text, ' '.join(new_matching))
        
        # Увеличиваем значимость лексического совпадения над семантическим
        total_score = (f_score * 0.7) + (similarity * 0.3)

        if total_score < 0.75:
            continue

        answer.append((matching, id, total_score))

    if answer:
        # Берем лучший результат
        best_match = max(answer, key=lambda x: x[2])
        id = best_match[1]
        concept = concepts[id][0]
        
        if concept != except_concept:
            return f'[{original_text}]({concept}.md)'
            
    return original_text

In [10]:
print(find_related_concept('физическую систему', ''))
print(find_related_concept('силы Лоренцаа)', ''))

физическую систему
[силы Лоренцаа)](Q172137.md)


In [11]:
import re

def add_links(markdown_text: str, except_concept: str) -> str:
    """
    Проходит по Markdown тексту и добавляет ссылки на концепты,
    используя жадный поиск (3-2-1 слова) и соблюдая границы пунктуации.
    """
    lines = markdown_text.split('\n')
    processed_lines = []

    for line in lines:
        # Правило 1: Не искать ссылки в заголовках
        if line.lstrip().startswith('#'):
            processed_lines.append(line)
            continue
        
        # Правило 2 & 4 & 5: Границы (.,:;). Разделяем, сохраняя сами разделители.
        # Это гарантирует, что поиск не выйдет за пределы части предложения.
        parts = re.split(r'([.,:;])', line)
        new_parts = []
        
        for part in parts:
            if part in '.,:;':
                new_parts.append(part)
            else:
                # Обрабатываем текст внутри «безопасного» сегмента
                new_parts.append(_process_segment(part, except_concept))
        
        processed_lines.append("".join(new_parts))
        
    return "\n".join(processed_lines)

def _process_segment(segment: str, except_concept: str) -> str:
    if not segment.strip():
        return segment
        
    # Разбиваем сегмент на токены: слова и пробельные последовательности между ними
    # Это позволяет нам точно восстановить строку с сохранением всех пробелов.
    tokens = re.split(r'(\s+)', segment)
    
    # Индексы токенов, которые содержат текст (не пробелы)
    word_indices = [i for i, t in enumerate(tokens) if t.strip()]
    
    result_fragments = []
    last_processed_token_idx = 0
    p = 0  # Указатель на текущее слово в списке word_indices
    
    while p < len(word_indices):
        found_match = False
        
        # Правило 4: Жадный поиск (сначала длинные, потом короткие)
        for window_size in [3, 2, 1]:
            if p + window_size <= len(word_indices):
                # Определяем диапазон токенов для проверки
                current_word_slice = word_indices[p : p + window_size]
                start_tok_idx = current_word_slice[0]
                end_tok_idx = current_word_slice[-1]
                
                # Собираем строку из токенов (включая пробелы между словами внутри окна)
                candidate = "".join(tokens[start_tok_idx : end_tok_idx + 1])
                
                # Пытаемся найти концепт
                linked = find_related_concept(candidate, except_concept)
                
                # Если функция вернула ссылку (строка изменилась)
                if linked != candidate:
                    # Добавляем всё, что было пропущено до этого момента (пробелы/слова без ссылок)
                    result_fragments.append("".join(tokens[last_processed_token_idx : start_tok_idx]))
                    # Добавляем саму ссылку
                    result_fragments.append(linked)
                    
                    last_processed_token_idx = end_tok_idx + 1
                    p += window_size
                    found_match = True
                    break
        
        if not found_match:
            # Если для текущего слова (и комбинаций с ним) ничего не найдено
            current_word_idx = word_indices[p]
            # Добавляем это слово в результат при следующей склейке или здесь:
            result_fragments.append("".join(tokens[last_processed_token_idx : current_word_idx + 1]))
            last_processed_token_idx = current_word_idx + 1
            p += 1
            
    # Добавляем остатки (например, пробелы в конце строки)
    result_fragments.append("".join(tokens[last_processed_token_idx:]))
    
    return "".join(result_fragments)

In [12]:
for concept_id, concept_name in tqdm(concepts):
    article = read_concept(concept_id)
    new_article = add_links(article, except_concept=concept_id)
    write_concept(concept_id, new_article)

100%|██████████| 168/168 [00:45<00:00,  3.68it/s]
